In [117]:
from pathlib import Path

print("Current Folder:", Path.cwd())

Current Folder: c:\Users\mdsih\OneDrive\Documents\Kaggle Project\causal-inference-bangladesh\notobooks


In [118]:
import pandas as pd

In [119]:
df = pd.read_csv("../data/BGD_2015_SCECSPIE-EL_v01_M_CSV/SIEF_endline_bayley.csv")


In [120]:
print("Rows and Columns:", df.shape)

Rows and Columns: (2486, 36)


In [121]:
df.columns.tolist()

['V001',
 'cc_code',
 'upazila',
 'district',
 'union',
 'treat',
 'tc',
 'TC_other',
 'V004',
 'V005',
 'V016',
 'V017',
 'V018',
 'V021_Y',
 'V021_M',
 'V022_M',
 'V023',
 'RC_1',
 'RC_2',
 'EC_1',
 'EC_2',
 'Sum_1',
 'Sum_2',
 'V024',
 'V025',
 'V026',
 'V027',
 'V028',
 'Cog_Total',
 'Rec_Total',
 'Exp_Total',
 'AM001',
 'AM002',
 'AM003',
 'AM004',
 'AM005']

In [122]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2486 entries, 0 to 2485
Data columns (total 36 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   V001       2486 non-null   int64  
 1   cc_code    2486 non-null   int64  
 2   upazila    2486 non-null   int64  
 3   district   2486 non-null   int64  
 4   union      2486 non-null   int64  
 5   treat      2486 non-null   int64  
 6   tc         2486 non-null   int64  
 7   TC_other   192 non-null    str    
 8   V004       2486 non-null   int64  
 9   V005       2486 non-null   int64  
 10  V016       2486 non-null   int64  
 11  V017       2486 non-null   int64  
 12  V018       2486 non-null   int64  
 13  V021_Y     2486 non-null   int64  
 14  V021_M     2485 non-null   float64
 15  V022_M     2486 non-null   int64  
 16  V023       2486 non-null   int64  
 17  RC_1       2484 non-null   float64
 18  RC_2       2484 non-null   float64
 19  EC_1       2485 non-null   float64
 20  EC_2       2484 non

In [123]:
df.isnull().sum()

V001            0
cc_code         0
upazila         0
district        0
union           0
treat           0
tc              0
TC_other     2294
V004            0
V005            0
V016            0
V017            0
V018            0
V021_Y          0
V021_M          1
V022_M          0
V023            0
RC_1            2
RC_2            2
EC_1            1
EC_2            2
Sum_1           2
Sum_2           4
V024            1
V025            1
V026            1
V027            1
V028            1
Cog_Total       0
Rec_Total       0
Exp_Total       0
AM001        2486
AM002        2486
AM003           0
AM004           0
AM005           0
dtype: int64

In [124]:
df["treat"].value_counts()

treat
1    1250
0    1236
Name: count, dtype: int64

In [125]:
df.head(3).T

,0,1,2
V001,1,2,3
cc_code,12,12,12
upazila,1,1,1
district,3,3,3
union,13,13,13
treat,1,1,1
tc,1,1,2
TC_other,NaN,NaN,NaN
V004,5,3,4
V005,2,1,2


In [126]:
df[["Cog_Total", "Rec_Total", "Exp_Total"]].describe()

,Cog_Total,Rec_Total,Exp_Total
count,2486.000000,2486.000000,2486.000000
mean,64.833467,30.182623,32.006034
std,5.313400,5.264478,4.693371
min,0.000000,0.000000,0.000000
25%,63.000000,26.000000,30.000000
50%,65.000000,30.000000,32.000000
75%,67.000000,34.000000,34.000000
max,81.000000,47.000000,47.000000


In [127]:
df.groupby("treat")[["Cog_Total", "Rec_Total", "Exp_Total"]].mean()

,Cog_Total,Rec_Total,Exp_Total
treat,,,
0,64.71521,29.814725,31.776699
1,64.95040,30.546400,32.232800


In [128]:
from scipy.stats import ttest_ind

for col in ["Cog_Total", "Rec_Total", "Exp_Total"]:
    control = df.loc[df["treat"] == 0, col]
    treatment = df.loc[df["treat"] == 1, col]

    stat, p_value = ttest_ind(treatment, control)
    print(f"{col}: p-value = {p_value:.4f}")

Cog_Total: p-value = 0.2699
Rec_Total: p-value = 0.0005
Exp_Total: p-value = 0.0154


In [129]:
df.groupby("treat")[["Cog_Total", "Rec_Total", "Exp_Total"]].agg(["mean", "std", "count"])

Cog_Total                  Rec_Total                  Exp_Total  \
           mean       std count       mean       std count       mean   
treat                                                                   
0      64.71521  4.965611  1236  29.814725  5.223628  1236  31.776699   
1      64.95040  5.635792  1250  30.546400  5.281492  1250  32.232800   

                       
            std count  
treat                  
0      4.580222  1236  
1      4.793682  1250

In [130]:
df["cc_code"].nunique()

78

In [131]:
df.groupby("treat")["cc_code"].nunique()

treat
0    39
1    39
Name: cc_code, dtype: int64

In [132]:
df.groupby("cc_code")["treat"].nunique().value_counts()

treat
1    78
Name: count, dtype: int64

In [133]:
df.groupby(["cc_code", "treat"]).size().describe()

count    78.000000
mean     31.871795
std       1.389749
min      27.000000
25%      31.000000
50%      32.000000
75%      33.000000
max      33.000000
dtype: float64

In [134]:
cluster_means = (
    df.groupby(["cc_code", "treat"])[
        ["Cog_Total", "Rec_Total", "Exp_Total"]
    ].mean()
)

cluster_means.head()

,,Cog_Total,Rec_Total,Exp_Total
cc_code,treat,,,
1,1,65.875000,32.218750,32.718750
2,1,64.787879,30.393939,32.484848
3,1,64.848485,31.969697,34.181818
4,0,65.375000,32.281250,33.281250
5,0,64.187500,30.562500,31.250000


In [135]:
from scipy.stats import ttest_ind

for col in ["Cog_Total", "Rec_Total", "Exp_Total"]:
    control = cluster_means.xs(0, level="treat")[col]
    treatment = cluster_means.xs(1, level="treat")[col]

    stat, p_value = ttest_ind(treatment, control)
    print(f"{col}: p-value = {p_value:.4f}")

Cog_Total: p-value = 0.5604
Rec_Total: p-value = 0.1004
Exp_Total: p-value = 0.1886


In [136]:
df.groupby("treat")[["V001", "cc_code"]].head(3)

,V001,cc_code
0,1,12
1,2,12
2,3,12
33,34,30
34,35,30
35,36,30


In [137]:
df.groupby("cc_code")["treat"].first().value_counts()

treat
1    39
0    39
Name: count, dtype: int64

In [138]:
df[["treat", "tc", "TC_other"]].value_counts(dropna=False).head(10)

treat  tc  TC_other           
0      1   NaN                    748
1      1   NaN                    637
       2   NaN                    414
0      2   NaN                    287
       3   NaN                    121
1      3   NaN                     83
       4   TEA GARDEN HOSPITAL     72
0      4   TEA GARDEN HOSPITAL     67
1      4   FWC                     44
0      4   FWC                      9
Name: count, dtype: int64

In [139]:
df["tc"].value_counts().sort_index()

tc
1    1385
2     701
3     204
4     196
Name: count, dtype: int64

In [140]:
pd.crosstab(df["tc"], df["treat"])

treat,0,1
tc,,
1,748,637
2,287,414
3,121,83
4,80,116


In [141]:
df.columns.tolist()

['V001',
 'cc_code',
 'upazila',
 'district',
 'union',
 'treat',
 'tc',
 'TC_other',
 'V004',
 'V005',
 'V016',
 'V017',
 'V018',
 'V021_Y',
 'V021_M',
 'V022_M',
 'V023',
 'RC_1',
 'RC_2',
 'EC_1',
 'EC_2',
 'Sum_1',
 'Sum_2',
 'V024',
 'V025',
 'V026',
 'V027',
 'V028',
 'Cog_Total',
 'Rec_Total',
 'Exp_Total',
 'AM001',
 'AM002',
 'AM003',
 'AM004',
 'AM005']

In [142]:
df[["RC_1", "RC_2", "EC_1", "EC_2", "Sum_1", "Sum_2", "Cog_Total", "Rec_Total", "Exp_Total"]].head()

,RC_1,RC_2,EC_1,EC_2,Sum_1,Sum_2,Cog_Total,Rec_Total,Exp_Total
0,25.0,7.0,29.0,7.0,14.0,83.0,63,25,29
1,28.0,8.0,28.0,7.0,15.0,86.0,55,28,28
2,32.0,8.0,33.0,7.0,15.0,86.0,64,32,33
3,33.0,11.0,33.0,10.0,21.0,103.0,66,33,33
4,23.0,7.0,26.0,7.0,14.0,83.0,57,23,26


In [143]:
df[["V001", "V004", "V005", "V016", "V017", "V018",
    "V021_Y", "V021_M", "V022_M", "V023"]].head()

,V001,V004,V005,V016,V017,V018,V021_Y,V021_M,V022_M,V023
0,1,5,2,63,7,85,2,8.0,32,15
1,2,3,1,55,4,70,2,8.0,32,15
2,3,4,2,64,6,80,3,1.0,37,16
3,4,3,2,66,9,95,2,5.0,29,15
4,5,5,2,57,6,80,2,3.0,27,14


In [144]:
df.isnull().sum()

V001            0
cc_code         0
upazila         0
district        0
union           0
treat           0
tc              0
TC_other     2294
V004            0
V005            0
V016            0
V017            0
V018            0
V021_Y          0
V021_M          1
V022_M          0
V023            0
RC_1            2
RC_2            2
EC_1            1
EC_2            2
Sum_1           2
Sum_2           4
V024            1
V025            1
V026            1
V027            1
V028            1
Cog_Total       0
Rec_Total       0
Exp_Total       0
AM001        2486
AM002        2486
AM003           0
AM004           0
AM005           0
dtype: int64

In [145]:
df[["Cog_Total", "Rec_Total", "Exp_Total"]].isnull().sum()

Cog_Total    0
Rec_Total    0
Exp_Total    0
dtype: int64

In [146]:
df.groupby("treat")[["Cog_Total", "Rec_Total", "Exp_Total"]].mean().round(3)

,Cog_Total,Rec_Total,Exp_Total
treat,,,
0,64.715,29.815,31.777
1,64.950,30.546,32.233


In [147]:
from scipy.stats import ttest_ind

for col in ["Cog_Total", "Rec_Total", "Exp_Total"]:
    control = df.loc[df["treat"] == 0, col]
    treatment = df.loc[df["treat"] == 1, col]

    t_stat, p_value = ttest_ind(
        treatment, control, equal_var=False
    )

    print(f"{col}: t-statistic={t_stat:.3f}, p-value={p_value:.4f}")

Cog_Total: t-statistic=1.104, p-value=0.2696
Rec_Total: t-statistic=3.473, p-value=0.0005
Exp_Total: t-statistic=2.426, p-value=0.0154


In [148]:
cluster_means = (
    df.groupby(["cc_code", "treat"])[
        ["Cog_Total", "Rec_Total", "Exp_Total"]
    ]
    .mean()
    .reset_index()
)

cluster_means.groupby("treat")[
    ["Cog_Total", "Rec_Total", "Exp_Total"]
].mean().round(3)

,Cog_Total,Rec_Total,Exp_Total
treat,,,
0,64.701,29.795,31.768
1,64.959,30.554,32.241


In [149]:
from scipy.stats import ttest_ind

for col in ["Cog_Total", "Rec_Total", "Exp_Total"]:
    control = cluster_means.loc[
        cluster_means["treat"] == 0, col
    ]
    treatment = cluster_means.loc[
        cluster_means["treat"] == 1, col
    ]

    t_stat, p_value = ttest_ind(
        treatment, control, equal_var=False
    )

    print(
        f"{col}: t-statistic={t_stat:.3f}, "
        f"p-value={p_value:.4f}"
    )

Cog_Total: t-statistic=0.585, p-value=0.5604
Rec_Total: t-statistic=1.663, p-value=0.1007
Exp_Total: t-statistic=1.327, p-value=0.1888


In [150]:
df.groupby("union")["cc_code"].nunique()

union
1     2
2     2
3     2
4     2
5     4
6     2
7     4
8     2
9     2
10    2
11    3
12    1
13    2
14    2
15    2
16    3
17    3
18    3
19    3
20    5
21    2
22    3
23    1
24    1
25    2
26    3
27    2
28    2
29    2
30    3
31    2
32    4
Name: cc_code, dtype: int64

In [151]:
pd.crosstab(
    df.drop_duplicates("cc_code")["union"],
    df.drop_duplicates("cc_code")["treat"]
)

treat,0,1
union,,
1,1,1
2,1,1
3,1,1
4,1,1
5,2,2
6,1,1
7,2,2
8,1,1
9,1,1


In [152]:
df.groupby("cc_code").size().describe().round(2)

count    78.00
mean     31.87
std       1.39
min      27.00
25%      31.00
50%      32.00
75%      33.00
max      33.00
dtype: float64

In [153]:
from scipy.stats import ttest_ind

for col in ["Cog_Total", "Rec_Total", "Exp_Total"]:
    control = cluster_means.loc[
        cluster_means["treat"] == 0, col
    ]
    treatment = cluster_means.loc[
        cluster_means["treat"] == 1, col
    ]

    result = ttest_ind(treatment, control, equal_var=False)
    ci = result.confidence_interval(confidence_level=0.95)

    print(
        f"{col}: Difference={treatment.mean() - control.mean():.3f}, "
        f"95% CI=({ci.low:.3f}, {ci.high:.3f})"
    )

Cog_Total: Difference=0.258, 95% CI=(-0.621, 1.137)
Rec_Total: Difference=0.759, 95% CI=(-0.151, 1.670)
Exp_Total: Difference=0.472, 95% CI=(-0.237, 1.182)


In [154]:
df.groupby("treat")["cc_code"].nunique()

treat
0    39
1    39
Name: cc_code, dtype: int64

In [155]:
%pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [156]:
import statsmodels.formula.api as smf

for outcome in ["Cog_Total", "Rec_Total", "Exp_Total"]:
    model = smf.ols(
        f"{outcome} ~ treat", data=df
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["cc_code"]}
    )

    print(f"\n{outcome}")
    print(f"Treatment effect: {model.params['treat']:.3f}")
    print(f"Cluster-robust p-value: {model.pvalues['treat']:.4f}")
    print(
        "95% CI:",
        model.conf_int().loc["treat"].round(3).tolist()
    )


Cog_Total
Treatment effect: 0.235
Cluster-robust p-value: 0.5890
95% CI: [-0.618, 1.088]

Rec_Total
Treatment effect: 0.732
Cluster-robust p-value: 0.1066
95% CI: [-0.157, 1.62]

Exp_Total
Treatment effect: 0.456
Cluster-robust p-value: 0.1966
95% CI: [-0.236, 1.148]


## Preliminary Results

Cluster-robust regression estimates show positive treatment effects
for all three child development outcomes.

- Cognitive score: 0.235 (p = 0.5890)
- Receptive communication: 0.732 (p = 0.1066)
- Expressive communication: 0.456 (p = 0.1966)

None of the estimated effects is statistically significant at the
5% level. These findings are preliminary and should be interpreted
in the context of the study's original randomization design.

In [157]:
df.groupby("union")["treat"].nunique().value_counts()

treat
2    29
1     3
Name: count, dtype: int64

In [158]:
df.groupby("union")["treat"].agg(["count", "mean"]).round(3)

,count,mean
union,,
1,64,0.500
2,65,0.508
3,65,0.508
4,66,0.500
5,130,0.500
6,57,0.491
7,127,0.512
8,64,0.516
9,63,0.508


In [159]:
df["union"].nunique(), df["upazila"].nunique(), df["district"].nunique()

(32, 3, 3)

In [160]:
df[["district", "upazila", "union"]].drop_duplicates().sort_values(
    ["district", "upazila", "union"]
).head(15)

,district,upazila,union
1123,1,2,6
1180,1,2,9
1031,1,2,17
1001,1,2,22
2262,2,3,1
1621,2,3,2
1555,2,3,4
1687,2,3,7
1817,2,3,8
1426,2,3,14


In [161]:
df.groupby(["district", "upazila"])["union"].nunique()

district  upazila
1         2           4
2         3          17
3         1          11
Name: union, dtype: int64

In [162]:
household = pd.read_csv(
    "../data/BGD_2015_SCECSPIE-EL_v01_M_CSV/SIEF_endline_household.csv"
)

household.shape

(2486, 562)

In [163]:
household.columns.tolist()[:40]

['V001',
 'cc_code',
 'treat',
 'hpi',
 'A001',
 'A008b',
 'A009',
 'A010',
 'A011',
 'Q02_B1',
 'Q03_B1',
 'Q05_B1',
 'Q06_B1',
 'Q07_B1',
 'Q08_B1',
 'Q09_B1',
 'Q10_B1',
 'Q11_B1',
 'Q12_B1',
 'Q13_B1',
 'Q14_B1',
 'HH_Info',
 'Q02_B2',
 'Q03_B2',
 'Q05_B2',
 'Q06_B2',
 'Q07_B2',
 'Q08_B2',
 'Q09_B2',
 'Q10_B2',
 'Q11_B2',
 'Q12_B2',
 'Q13_B2',
 'Q14_B2',
 'Q02_B3',
 'Q03_B3',
 'Q05_B3',
 'Q06_B3',
 'Q07_B3',
 'Q08_B3']

In [164]:
household[["V001", "cc_code", "treat", "hpi"]].head()

,V001,cc_code,treat,hpi
0,535,1,1,6
1,541,1,1,8
2,548,1,1,2
3,550,1,1,9
4,555,1,1,3


In [165]:
household["hpi"].value_counts(dropna=False).sort_index()

hpi
2      43
3     541
4     568
5     608
6     346
7     159
8      82
9      79
10     25
11     13
12     14
13      2
14      4
16      1
17      1
Name: count, dtype: int64

In [166]:
household["hpi"].agg(["count", "nunique", "min", "max"])

count      2486
nunique      15
min           2
max          17
Name: hpi, dtype: int64

In [167]:
household[["V001", "cc_code", "treat"]].duplicated().sum()

np.int64(0)

In [168]:
print("Bayley unique V001:", df["V001"].nunique())
print("Household unique V001:", household["V001"].nunique())
print("Common V001:", df["V001"].isin(household["V001"]).sum())

Bayley unique V001: 2486
Household unique V001: 2486
Common V001: 2486


In [169]:
check = df[["V001", "cc_code", "treat"]].merge(
    household[["V001", "cc_code", "treat"]],
    on="V001",
    suffixes=("_bayley", "_household")
)

print("cc_code match:", (
    check["cc_code_bayley"] == check["cc_code_household"]
).all())

print("treat match:", (
    check["treat_bayley"] == check["treat_household"]
).all())

cc_code match: True
treat match: True


In [170]:
merged_df = df.merge(
    household,
    on=["V001", "cc_code", "treat"],
    how="left",
    validate="one_to_one",
    suffixes=("_bayley", "_household")
)

print("Merged shape:", merged_df.shape)
print("Rows preserved:", len(merged_df) == len(df))

Merged shape: (2486, 595)
Rows preserved: True


In [171]:
merged_df.isnull().sum().sort_values(ascending=False).head(15)

R004       2486
AM002      2486
M001e_m    2486
M001e_y    2486
M001e_2    2486
AM001      2486
M001e_3    2486
A001       2486
Q13_B20    2486
Q14_B20    2486
Q12_B20    2486
I002h      2486
B027       2486
Q12_B19    2486
Q13_B19    2486
dtype: int64

In [172]:
merged_df.notnull().sum().sort_values(ascending=False).head(20)

district_household    2486
V001                  2486
cc_code               2486
P001                  2486
AM004                 2486
AM003                 2486
Exp_Total             2486
Rec_Total             2486
Cog_Total             2486
O001                  2486
N001                  2486
M014                  2486
M001e_1               2486
L006                  2486
L005                  2486
L004                  2486
L003                  2486
L002                  2486
L001                  2486
K015                  2486
dtype: int64

In [173]:
household.dtypes.head(40)

V001         int64
cc_code      int64
treat        int64
hpi          int64
A001       float64
A008b        int64
A009       float64
A010         int64
A011         int64
Q02_B1     float64
Q03_B1     float64
Q05_B1     float64
Q06_B1     float64
Q07_B1     float64
Q08_B1     float64
Q09_B1     float64
Q10_B1     float64
Q11_B1     float64
Q12_B1     float64
Q13_B1     float64
Q14_B1     float64
HH_Info      int64
Q02_B2     float64
Q03_B2     float64
Q05_B2     float64
Q06_B2     float64
Q07_B2     float64
Q08_B2     float64
Q09_B2     float64
Q10_B2     float64
Q11_B2     float64
Q12_B2     float64
Q13_B2     float64
Q14_B2     float64
Q02_B3     float64
Q03_B3     float64
Q05_B3     float64
Q06_B3     float64
Q07_B3     float64
Q08_B3     float64
dtype: object

In [174]:
household.columns.tolist()

['V001',
 'cc_code',
 'treat',
 'hpi',
 'A001',
 'A008b',
 'A009',
 'A010',
 'A011',
 'Q02_B1',
 'Q03_B1',
 'Q05_B1',
 'Q06_B1',
 'Q07_B1',
 'Q08_B1',
 'Q09_B1',
 'Q10_B1',
 'Q11_B1',
 'Q12_B1',
 'Q13_B1',
 'Q14_B1',
 'HH_Info',
 'Q02_B2',
 'Q03_B2',
 'Q05_B2',
 'Q06_B2',
 'Q07_B2',
 'Q08_B2',
 'Q09_B2',
 'Q10_B2',
 'Q11_B2',
 'Q12_B2',
 'Q13_B2',
 'Q14_B2',
 'Q02_B3',
 'Q03_B3',
 'Q05_B3',
 'Q06_B3',
 'Q07_B3',
 'Q08_B3',
 'Q09_B3',
 'Q10_B3',
 'Q11_B3',
 'Q12_B3',
 'Q13_B3',
 'Q14_B3',
 'Q02_B4',
 'Q03_B4',
 'Q05_B4',
 'Q06_B4',
 'Q07_B4',
 'Q08_B4',
 'Q09_B4',
 'Q10_B4',
 'Q11_B4',
 'Q12_B4',
 'Q13_B4',
 'Q14_B4',
 'Q02_B5',
 'Q03_B5',
 'Q05_B5',
 'Q06_B5',
 'Q07_B5',
 'Q08_B5',
 'Q09_B5',
 'Q10_B5',
 'Q11_B5',
 'Q12_B5',
 'Q13_B5',
 'Q14_B5',
 'Q02_B6',
 'Q03_B6',
 'Q05_B6',
 'Q06_B6',
 'Q07_B6',
 'Q08_B6',
 'Q09_B6',
 'Q10_B6',
 'Q11_B6',
 'Q12_B6',
 'Q13_B6',
 'Q14_B6',
 'Q02_B7',
 'Q03_B7',
 'Q05_B7',
 'Q06_B7',
 'Q07_B7',
 'Q08_B7',
 'Q09_B7',
 'Q10_B7',
 'Q11_B7',
 'Q12_B7',
 

In [175]:
household[["A001", "A008b", "A009", "A010", "A011"]].head(10)

,A001,A008b,A009,A010,A011
0,NaN,2,2.0,30,1
1,NaN,3,2.5,20,4
2,NaN,3,1.0,20,2
3,NaN,3,1.5,25,2
4,NaN,3,3.0,20,2
5,NaN,3,1.5,20,2
6,NaN,3,3.0,60,1
7,NaN,3,3.0,60,1
8,NaN,3,2.0,30,2
9,NaN,3,3.0,60,1


In [176]:
household[["A001", "A008b", "A009", "A010", "A011"]].isnull().sum()

A001     2486
A008b       0
A009        0
A010        0
A011        0
dtype: int64

In [177]:
[c for c in ["A008b", "A009", "A010", "A011", "A012"] if c in household.columns]

['A008b', 'A009', 'A010', 'A011']

In [178]:
household[["A008b", "A009", "A010", "A011"]].describe()

,A008b,A009,A010,A011
count,2486.000000,2486.000000,2486.000000,2486.000000
mean,2.224055,1.742949,30.147224,1.319791
std,0.613136,1.144944,18.787685,0.690483
min,1.000000,0.000000,0.000000,1.000000
25%,2.000000,1.000000,15.000000,1.000000
50%,2.000000,1.500000,30.000000,1.000000
75%,3.000000,2.250000,40.000000,1.000000
max,3.000000,9.000000,130.000000,5.000000


In [179]:
for col in ["A008b", "A009", "A010", "A011"]:
    print(f"\n{col}:")
    print(household[col].value_counts().sort_index())


A008b:
A008b
1     251
2    1427
3     808
Name: count, dtype: int64

A009:
A009
0.00     68
0.05      2
0.10      7
0.15      4
0.20     12
0.22      1
0.25     92
0.30     12
0.35      2
0.40     13
0.50    221
0.60      5
0.65      1
0.70      5
0.75     30
0.80      6
0.90      1
1.00    422
1.10      2
1.15      1
1.20     12
1.25     22
1.30     14
1.40      7
1.50    370
1.60      4
1.65      1
1.70      7
1.75     13
1.80     13
1.90      4
2.00    470
2.10      1
2.20      7
2.25     18
2.30      3
2.40      3
2.50    162
2.60      1
2.70      1
2.75      9
2.80      7
3.00    240
3.20      1
3.25      2
3.30      2
3.50     38
3.80      2
4.00     70
4.50      7
5.00     48
5.50      1
6.00     11
7.00      5
8.00      2
9.00      1
Name: count, dtype: int64

A010:
A010
0        3
1        8
2       24
3       16
4       12
5       74
6        9
7       20
8       25
9        4
10     241
11       2
12      20
13       5
14       1
15     204
16       2
17       8
18       5

In [180]:
print("A010:")
print(household["A010"].value_counts(dropna=False).sort_index())

print("\nA011:")
print(household["A011"].value_counts(dropna=False).sort_index())

A010:
A010
0        3
1        8
2       24
3       16
4       12
5       74
6        9
7       20
8       25
9        4
10     241
11       2
12      20
13       5
14       1
15     204
16       2
17       8
18       5
20     317
22       8
25     194
28       2
30     358
35     139
37       1
38       2
40     296
41       1
44       1
45      74
46       2
48       1
50     128
52       1
54       1
55      18
57       1
59       1
60     134
65      20
70      40
75      15
80      14
84       2
85       1
90      15
100      7
120      8
130      1
Name: count, dtype: int64

A011:
A011
1    1905
2     474
3       1
4     105
5       1
Name: count, dtype: int64


In [181]:
print("A010 unique values:", household["A010"].nunique())
print("A011 distribution:")
print(household["A011"].value_counts(dropna=False).sort_index().to_string())

A010 unique values: 50
A011 distribution:
A011
1    1905
2     474
3       1
4     105
5       1


In [182]:
print(household.columns[:100].tolist())

['V001', 'cc_code', 'treat', 'hpi', 'A001', 'A008b', 'A009', 'A010', 'A011', 'Q02_B1', 'Q03_B1', 'Q05_B1', 'Q06_B1', 'Q07_B1', 'Q08_B1', 'Q09_B1', 'Q10_B1', 'Q11_B1', 'Q12_B1', 'Q13_B1', 'Q14_B1', 'HH_Info', 'Q02_B2', 'Q03_B2', 'Q05_B2', 'Q06_B2', 'Q07_B2', 'Q08_B2', 'Q09_B2', 'Q10_B2', 'Q11_B2', 'Q12_B2', 'Q13_B2', 'Q14_B2', 'Q02_B3', 'Q03_B3', 'Q05_B3', 'Q06_B3', 'Q07_B3', 'Q08_B3', 'Q09_B3', 'Q10_B3', 'Q11_B3', 'Q12_B3', 'Q13_B3', 'Q14_B3', 'Q02_B4', 'Q03_B4', 'Q05_B4', 'Q06_B4', 'Q07_B4', 'Q08_B4', 'Q09_B4', 'Q10_B4', 'Q11_B4', 'Q12_B4', 'Q13_B4', 'Q14_B4', 'Q02_B5', 'Q03_B5', 'Q05_B5', 'Q06_B5', 'Q07_B5', 'Q08_B5', 'Q09_B5', 'Q10_B5', 'Q11_B5', 'Q12_B5', 'Q13_B5', 'Q14_B5', 'Q02_B6', 'Q03_B6', 'Q05_B6', 'Q06_B6', 'Q07_B6', 'Q08_B6', 'Q09_B6', 'Q10_B6', 'Q11_B6', 'Q12_B6', 'Q13_B6', 'Q14_B6', 'Q02_B7', 'Q03_B7', 'Q05_B7', 'Q06_B7', 'Q07_B7', 'Q08_B7', 'Q09_B7', 'Q10_B7', 'Q11_B7', 'Q12_B7', 'Q13_B7', 'Q14_B7', 'Q02_B8', 'Q03_B8', 'Q05_B8', 'Q06_B8', 'Q07_B8', 'Q08_B8']


In [183]:
print(household.columns[100:160].tolist())

['Q09_B8', 'Q10_B8', 'Q11_B8', 'Q12_B8', 'Q13_B8', 'Q14_B8', 'Q02_B9', 'Q03_B9', 'Q05_B9', 'Q06_B9', 'Q07_B9', 'Q08_B9', 'Q09_B9', 'Q10_B9', 'Q11_B9', 'Q12_B9', 'Q13_B9', 'Q14_B9', 'Q02_B10', 'Q03_B10', 'Q05_B10', 'Q06_B10', 'Q07_B10', 'Q08_B10', 'Q09_B10', 'Q10_B10', 'Q11_B10', 'Q12_B10', 'Q13_B10', 'Q14_B10', 'Q02_B11', 'Q03_B11', 'Q05_B11', 'Q06_B11', 'Q07_B11', 'Q08_B11', 'Q09_B11', 'Q10_B11', 'Q11_B11', 'Q12_B11', 'Q13_B11', 'Q14_B11', 'Q02_B12', 'Q03_B12', 'Q05_B12', 'Q06_B12', 'Q07_B12', 'Q08_B12', 'Q09_B12', 'Q10_B12', 'Q11_B12', 'Q12_B12', 'Q13_B12', 'Q14_B12', 'Q02_B13', 'Q03_B13', 'Q05_B13', 'Q06_B13', 'Q07_B13', 'Q08_B13']


In [184]:
print(household.columns[160:].tolist())

['Q09_B13', 'Q10_B13', 'Q11_B13', 'Q12_B13', 'Q13_B13', 'Q14_B13', 'Q02_B14', 'Q03_B14', 'Q05_B14', 'Q06_B14', 'Q07_B14', 'Q08_B14', 'Q09_B14', 'Q10_B14', 'Q11_B14', 'Q12_B14', 'Q13_B14', 'Q14_B14', 'Q02_B15', 'Q03_B15', 'Q05_B15', 'Q06_B15', 'Q07_B15', 'Q08_B15', 'Q09_B15', 'Q10_B15', 'Q11_B15', 'Q12_B15', 'Q13_B15', 'Q14_B15', 'Q02_B16', 'Q03_B16', 'Q05_B16', 'Q06_B16', 'Q07_B16', 'Q08_B16', 'Q09_B16', 'Q10_B16', 'Q11_B16', 'Q12_B16', 'Q13_B16', 'Q14_B16', 'Q02_B17', 'Q03_B17', 'Q05_B17', 'Q06_B17', 'Q07_B17', 'Q08_B17', 'Q09_B17', 'Q10_B17', 'Q11_B17', 'Q12_B17', 'Q13_B17', 'Q14_B17', 'Q02_B18', 'Q03_B18', 'Q05_B18', 'Q06_B18', 'Q07_B18', 'Q08_B18', 'Q09_B18', 'Q10_B18', 'Q11_B18', 'Q12_B18', 'Q13_B18', 'Q14_B18', 'Q02_B19', 'Q03_B19', 'Q05_B19', 'Q06_B19', 'Q07_B19', 'Q08_B19', 'Q09_B19', 'Q10_B19', 'Q11_B19', 'Q12_B19', 'Q13_B19', 'Q14_B19', 'Q02_B20', 'Q03_B20', 'Q05_B20', 'Q06_B20', 'Q07_B20', 'Q08_B20', 'Q09_B20', 'Q10_B20', 'Q11_B20', 'Q12_B20', 'Q13_B20', 'Q14_B20', 'HH_Head_

In [185]:
cols = [
    "B013", "B014", "B014a", "B014b",
    "B015_1", "B016_1", "B017_1",
    "B019_1", "B020", "B021", "B022",
    "B023", "B024", "B025", "B026", "B027"
]

summary = pd.DataFrame({
    "missing": household[cols].isnull().sum(),
    "unique_values": household[cols].nunique(),
    "dtype": household[cols].dtypes.astype(str)
})

summary

,missing,unique_values,dtype
B013,0,70,int64
B014,1712,3,float64
B014a,0,2,int64
B014b,2391,1,float64
B015_1,2391,51,float64
B016_1,2391,2,float64
B017_1,2391,2,float64
B019_1,2391,7,float64
B020,0,2,int64
B021,0,2,int64


In [186]:
cols = [
    "C001", "C002", "C003", "C004",
    "C005a", "C005b", "C005c", "C005d",
    "C005e", "C005f", "C005g", "C005h",
    "C005i", "C005j", "C005k", "C005l",
    "C005m", "C005n", "C005o", "C005p"
]

pd.DataFrame({
    "missing": household[cols].isnull().sum(),
    "unique_values": household[cols].nunique()
})

,missing,unique_values
C001,0,5
C002,0,7
C003,10,2
C004,0,7
C005a,0,2
C005b,0,2
C005c,0,2
C005d,0,2
C005e,0,2
C005f,0,2


In [187]:
for col in ["C001", "C002", "C003", "C004", "C005a", "C005b", "C005c"]:
    print(f"\n{col}:")
    print(household[col].value_counts(dropna=False).sort_index())


C001:
C001
1     850
2    1563
3      12
7       2
9      59
Name: count, dtype: int64

C002:
C002
1    390
2    574
3    923
4    105
5    467
6     17
7     10
Name: count, dtype: int64

C003:
C003
1.0    2372
2.0     104
NaN      10
Name: count, dtype: int64

C004:
C004
1     459
2    1098
3     764
4      88
5      61
6      15
7       1
Name: count, dtype: int64

C005a:
C005a
1      74
2    2412
Name: count, dtype: int64

C005b:
C005b
1      55
2    2431
Name: count, dtype: int64

C005c:
C005c
1     329
2    2157
Name: count, dtype: int64


In [188]:
print(household["C004"].value_counts().sort_index())

C004
1     459
2    1098
3     764
4      88
5      61
6      15
7       1
Name: count, dtype: int64


In [189]:
asset_cols = [
    "C005a", "C005b", "C005c", "C005d",
    "C005e", "C005f", "C005g", "C005h",
    "C005i", "C005j", "C005k", "C005l",
    "C005m", "C005n", "C005o", "C005p"
]

household[asset_cols].apply(lambda col: sorted(col.dropna().unique()))

,C005a,C005b,C005c,C005d,C005e,C005f,C005g,C005h,C005i,C005j,C005k,C005l,C005m,C005n,C005o,C005p
0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2


In [190]:
asset_cols = [
    "C005a", "C005b", "C005c", "C005d",
    "C005e", "C005f", "C005g", "C005h",
    "C005i", "C005j", "C005k", "C005l",
    "C005m", "C005n", "C005o", "C005p"
]

household["asset_score"] = (household[asset_cols] == 1).sum(axis=1)

household["asset_score"].describe()

C:\Users\mdsih\AppData\Local\Temp\ipykernel_3408\365539091.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  household["asset_score"] = (household[asset_cols] == 1).sum(axis=1)


count    2486.000000
mean        5.928399
std         2.577511
min         0.000000
25%         4.000000
50%         6.000000
75%         8.000000
max        15.000000
Name: asset_score, dtype: float64

In [191]:
household.groupby("treat")["asset_score"].agg(
    ["count", "mean", "std", "median"]
)

,count,mean,std,median
treat,,,,
0,1236,5.919903,2.582888,6.0
1,1250,5.936800,2.573189,6.0


In [192]:
print("Number of clusters:", df["cc_code"].nunique())

print("\nClusters by treatment:")
print(
    df.groupby("treat")["cc_code"]
      .nunique()
)

print("\nTreatment assignment within clusters:")
print(
    df.groupby("cc_code")["treat"]
      .nunique()
      .value_counts()
)

Number of clusters: 78

Clusters by treatment:
treat
0    39
1    39
Name: cc_code, dtype: int64

Treatment assignment within clusters:
treat
1    78
Name: count, dtype: int64


In [193]:
cluster_summary = (
    df.groupby(["cc_code", "treat"])
      .agg(
          n_children=("V001", "count"),
          mean_cog=("Cog_Total", "mean"),
          mean_rec=("Rec_Total", "mean"),
          mean_exp=("Exp_Total", "mean")
      )
      .reset_index()
)

cluster_summary.groupby("treat")["n_children"].describe()

,count,mean,std,min,25%,50%,75%,max
treat,,,,,,,,
0,39.0,31.692308,1.259780,29.0,31.0,32.0,33.0,33.0
1,39.0,32.051282,1.503482,27.0,32.0,33.0,33.0,33.0


In [194]:
import pandas as pd

results = pd.DataFrame({
    "Outcome": [
        "Cognitive score",
        "Receptive communication",
        "Expressive communication"
    ],
    "Treatment Effect": [
        0.235,
        0.732,
        0.456
    ],
    "P-value": [
        0.5890,
        0.1066,
        0.1966
    ],
    "CI Lower": [
        -0.618,
        -0.157,
        -0.236
    ],
    "CI Upper": [
        1.088,
        1.620,
        1.148
    ]
})

results.round(3)

,Outcome,Treatment Effect,P-value,CI Lower,CI Upper
0,Cognitive score,0.235,0.589,-0.618,1.088
1,Receptive communication,0.732,0.107,-0.157,1.620
2,Expressive communication,0.456,0.197,-0.236,1.148


Preliminary Treatment Effect Results

The preliminary unadjusted OLS regressions use cluster-robust standard errors at the community clinic level.

All three estimated treatment effects are positive.
None is statistically significant at the 5% level.
The confidence intervals include zero for all three outcomes.
These estimates are preliminary and do not adjust for baseline outcomes.

The final analysis should account for the original randomization design and compare results with the published study's ANCOVA estimates.

In [195]:
df[["Cog_Total", "Rec_Total", "Exp_Total"]].describe()

,Cog_Total,Rec_Total,Exp_Total
count,2486.000000,2486.000000,2486.000000
mean,64.833467,30.182623,32.006034
std,5.313400,5.264478,4.693371
min,0.000000,0.000000,0.000000
25%,63.000000,26.000000,30.000000
50%,65.000000,30.000000,32.000000
75%,67.000000,34.000000,34.000000
max,81.000000,47.000000,47.000000


In [196]:
df.groupby("treat")[["Cog_Total", "Rec_Total", "Exp_Total"]].agg(
    ["count", "mean", "std"]
).round(3)

Cog_Total                Rec_Total                Exp_Total          \
          count    mean    std     count    mean    std     count    mean   
treat                                                                       
0          1236  64.715  4.966      1236  29.815  5.224      1236  31.777   
1          1250  64.950  5.636      1250  30.546  5.281      1250  32.233   

              
         std  
treat         
0      4.580  
1      4.794

In [197]:
for col in ["Cog_Total", "Rec_Total", "Exp_Total"]:
    print(f"\n{col}")
    print("Zero scores:", (df[col] == 0).sum())
    print("Missing values:", df[col].isna().sum())


Cog_Total
Zero scores: 2
Missing values: 0

Rec_Total
Zero scores: 2
Missing values: 0

Exp_Total
Zero scores: 1
Missing values: 0


In [198]:
attrition = pd.read_csv(
    "../data/BGD_2015_SCECSPIE-EL_v01_M_CSV/SIEF_endline_attrition.csv"
)

print("Shape:", attrition.shape)
print(attrition.columns.tolist())
print(attrition.head())

Shape: (88, 9)
['V001', 'upazila', 'district', 'union', 'cc_code', 'treat', 'HH_P_ID', 'A003', 'A006']
   V001  upazila  district  union  cc_code  treat  HH_P_ID  A003  A006
0   530        1         3      5        1      1        5     1   3.0
1   834        1         3      3        4      0        7     2   NaN
2   577        1         3     12        5      0        6     4  10.0
3   809        1         3     16        6      1        6     1   1.0
4   808        1         3     16        6      1        7     1   1.0


In [199]:
print("A003 values:")
print(attrition["A003"].value_counts(dropna=False).sort_index())

print("\nA006 values:")
print(attrition["A006"].value_counts(dropna=False).sort_index())

A003 values:
A003
1    68
2    12
3     3
4     4
5     1
Name: count, dtype: int64

A006 values:
A006
1.0     14
2.0     35
3.0      8
4.0      3
5.0      1
6.0      2
7.0      2
8.0      1
9.0      1
10.0     5
11.0     1
NaN     15
Name: count, dtype: int64


In [200]:
print("Attrition file treatment counts:")
print(attrition["treat"].value_counts())

print("\nAttrition file unique children:", attrition["V001"].nunique())

print("\nBayley file unique children:", df["V001"].nunique())

print("\nAttrition IDs found in Bayley:", 
      attrition["V001"].isin(df["V001"]).sum())

Attrition file treatment counts:
treat
0    51
1    37
Name: count, dtype: int64

Attrition file unique children: 88

Bayley file unique children: 2486

Attrition IDs found in Bayley: 0


In [201]:
df[["Cog_Total", "Rec_Total", "Exp_Total"]].head(10)

,Cog_Total,Rec_Total,Exp_Total
0,63,25,29
1,55,28,28
2,64,32,33
3,66,33,33
4,57,23,26
5,65,28,30
6,65,29,32
7,59,25,29
8,64,32,31
9,77,44,46


In [202]:
print(df.columns.tolist())

['V001', 'cc_code', 'upazila', 'district', 'union', 'treat', 'tc', 'TC_other', 'V004', 'V005', 'V016', 'V017', 'V018', 'V021_Y', 'V021_M', 'V022_M', 'V023', 'RC_1', 'RC_2', 'EC_1', 'EC_2', 'Sum_1', 'Sum_2', 'V024', 'V025', 'V026', 'V027', 'V028', 'Cog_Total', 'Rec_Total', 'Exp_Total', 'AM001', 'AM002', 'AM003', 'AM004', 'AM005']


In [203]:
df[["RC_1", "RC_2", "EC_1", "EC_2", "Sum_1", "Sum_2",
    "Cog_Total", "Rec_Total", "Exp_Total"]].describe().round(2)

,RC_1,RC_2,EC_1,EC_2,Sum_1,Sum_2,Cog_Total,Rec_Total,Exp_Total
count,2484.00,2484.00,2485.00,2484.00,2484.00,2482.00,2486.00,2486.00,2486.00
mean,30.21,8.53,32.02,7.83,16.40,89.18,64.83,30.18,32.01
std,5.20,2.70,4.65,1.73,4.24,11.45,5.31,5.26,4.69
min,7.00,1.00,11.00,2.00,3.00,3.00,0.00,0.00,0.00
25%,26.00,7.00,30.00,7.00,14.00,83.00,63.00,26.00,30.00
50%,30.00,8.00,32.00,8.00,16.00,89.00,65.00,30.00,32.00
75%,34.00,10.00,34.00,9.00,18.00,94.00,67.00,34.00,34.00
max,47.00,98.00,47.00,17.00,89.00,200.00,81.00,47.00,47.00


In [204]:
df.loc[
    (df["RC_2"] > 20) | (df["Sum_1"] > 30) | (df["Sum_2"] > 120),
    ["V001", "RC_1", "RC_2", "EC_1", "EC_2",
     "Sum_1", "Sum_2", "Cog_Total", "Rec_Total", "Exp_Total"]
].head(10)

,V001,RC_1,RC_2,EC_1,EC_2,Sum_1,Sum_2,Cog_Total,Rec_Total,Exp_Total
57,58,40.0,14.0,42.0,13.0,27.0,121.0,66,40,42
111,113,46.0,15.0,47.0,13.0,28.0,124.0,76,46,47
230,233,44.0,15.0,43.0,12.0,27.0,121.0,71,44,43
312,319,45.0,17.0,44.0,13.0,30.0,129.0,75,45,44
470,482,45.0,15.0,45.0,12.0,27.0,121.0,76,45,45
560,573,42.0,14.0,44.0,13.0,27.0,121.0,73,42,44
651,667,44.0,16.0,43.0,13.0,29.0,127.0,73,44,43
664,680,43.0,15.0,44.0,13.0,28.0,124.0,74,43,44
675,691,44.0,15.0,45.0,13.0,28.0,124.0,74,44,45
784,802,35.0,9.0,33.0,7.0,89.0,7.0,74,35,33


In [205]:
import statsmodels.formula.api as smf

for outcome in ["Cog_Total", "Rec_Total", "Exp_Total"]:
    model = smf.ols(
        f"{outcome} ~ treat",
        data=df
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": df["cc_code"]}
    )

    print(f"\n{outcome}")
    print(model.summary().tables[1])


Cog_Total
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     64.7152      0.319    202.620      0.000      64.089      65.341
treat          0.2352      0.435      0.540      0.589      -0.618       1.088

Rec_Total
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     29.8147      0.364     81.819      0.000      29.101      30.529
treat          0.7317      0.453      1.614      0.107      -0.157       1.620

Exp_Total
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     31.7767      0.274    116.152      0.000      31.240      32.313
treat          0.4561      0.353      1.291      0.197      -0.236       1.148
